# RAG Chat + TTS (Google Colab)

A retrieval-augmented chat pipeline with voice output:
- Retrieval: `sentence-transformers` + `FAISS`
- Generation: `Qwen2.5-1.5B-Instruct` (free, local via Hugging Face)
- Voice output: `gTTS`
- Chat UI: `Gradio`

For GPU acceleration: `Runtime → Change runtime type → GPU`.


In [ ]:
!pip -q install transformers accelerate sentence-transformers faiss-cpu gradio gTTS


In [ ]:
import re
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from gtts import gTTS
import gradio as gr


## Document input

Paste your text below, or use the file upload cell instead.


In [ ]:
DOCUMENT_TEXT = """
Paste your document text here.
It can span multiple paragraphs.
"""

print("Loaded chars:", len(DOCUMENT_TEXT))


### Alternative: upload a .txt file

Uncomment and run to upload a file instead of pasting text.


In [ ]:
# from google.colab import files
# uploaded = files.upload()
# fname = list(uploaded.keys())[0]
# with open(fname, "r", encoding="utf-8", errors="ignore") as f:
#     DOCUMENT_TEXT = f.read()
# print("Loaded chars:", len(DOCUMENT_TEXT))


## Chunking, embedding, and indexing


In [ ]:
def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def chunk_text(text: str, chunk_size=650, overlap=120):
    text = clean_text(text)
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunks = chunk_text(DOCUMENT_TEXT)
print("Chunks:", len(chunks))

emb = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=True)
emb = np.array(emb).astype("float32")

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

print("Index size:", index.ntotal)


## Load the language model


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto"
)

gen = pipeline("text-generation", model=model, tokenizer=tokenizer)

print("Model loaded:", MODEL_NAME)


## RAG pipeline and TTS


In [ ]:
def retrieve(query: str, top_k=4):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    q_emb = np.array(q_emb).astype("float32")
    _, ids = index.search(q_emb, top_k)
    return [chunks[i] for i in ids[0] if i != -1]

SYSTEM_RULES = (
    "You are a helpful assistant. Answer ONLY using the provided context from the document. "
    "If the answer is not in the context, say: 'I cannot find that in the document.'"
)

def build_prompt(contexts, question: str):
    context_block = "\n\n---\n\n".join(contexts)
    return f"""System: {SYSTEM_RULES}

Context:
{context_block}

User question: {question}

Assistant:"""

def answer_question(question: str, top_k=4, max_new_tokens=220):
    ctx = retrieve(question, top_k=top_k)
    prompt = build_prompt(ctx, question)

    out = gen(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.4,
        top_p=0.9,
        repetition_penalty=1.05
    )[0]["generated_text"]

    answer = out.split("Assistant:", 1)[-1].strip()
    return answer, ctx

def tts_to_file(text: str, path="answer.mp3", lang="en"):
    gTTS(text=text, lang=lang).save(path)
    return path


## Chat interface (Gradio)


In [ ]:
def chat_fn(message, history):
    answer, ctx = answer_question(message, top_k=4)

    lang = "fa" if re.search(r"[\u0600-\u06FF]", answer) else "en"
    audio_path = tts_to_file(answer[:800], lang=lang)

    sources = "\n\n".join(f"- {c}" for c in ctx)
    reply = f"{answer}\n\nRetrieved context:\n{sources}"
    return reply, audio_path

with gr.Blocks() as demo:
    gr.Markdown("## RAG Chat + TTS")
    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Your question")
    audio = gr.Audio(label="Answer (TTS)", type="filepath")
    clear = gr.Button("Clear")

    def user_submit(user_message, history):
        return "", history + [[user_message, None]]

    def bot_respond(history):
        user_message = history[-1][0]
        reply, audio_path = chat_fn(user_message, history[:-1])
        history[-1][1] = reply
        return history, audio_path

    msg.submit(user_submit, [msg, chatbot], [msg, chatbot]).then(
        bot_respond, [chatbot], [chatbot, audio]
    )
    clear.click(lambda: ([], None), outputs=[chatbot, audio])

demo.launch(share=True)
